# Обучение моделей

## Схема валидации (TimeSeriesSplit, no leakage)
```
Fold 1: Oct-Feb → Val Mar
Fold 2: Oct-Mar → Val Apr  
Fold 3: Oct-Apr → Val May   ← самый важный, ближе к test
```

## Стратегия обучения:
- Позитив: target=1 (51,438 фродовых операций)
- Негатив: target=0 (36,076 подтверждённых) + **сэмплированные null**
- Null-класс слишком большой (28M+) → берём 5-10x от позитива

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from pathlib import Path
import gc, joblib

ROOT = Path('/home/vadim/PyPr/hak')
FEATURES_OUT = ROOT / 'features'
MODELS_OUT = ROOT / 'models'
MODELS_OUT.mkdir(exist_ok=True)

# Из 02_features.ipynb
FEATURE_COLS = [
    'hour', 'weekday', 'month', 'is_night',
    'log_amount', 'operaton_amt', 'is_null_amount',
    'phone_voip_call_state', 'web_rdp_connection', 'compromised',
    'developer_tools', 'security_flags_sum',
    'event_type_nm', 'is_high_risk_type',
    'mcc_code', 'is_null_mcc',
    'channel_indicator_type', 'channel_indicator_sub_type', 'currency_iso_cd',
    'battery', 'operating_system_type', 'pos_cd',
    'cnt_1h', 'cnt_6h', 'cnt_24h', 'cnt_7d', 'cnt_30d',
    'amt_sum_1h', 'amt_sum_6h', 'amt_sum_24h', 'amt_sum_7d', 'amt_sum_30d',
    'secs_since_last', 'voip_cnt_24h',
    'is_new_mcc_code', 'is_new_channel_indicator_type', 'is_new_currency_iso_cd',
    'cum_unique_mcc_approx',
    'session_ops_before', 'session_amt_before',
    'timezone',
]

print(f'Features: {len(FEATURE_COLS)}')

## 1. Загрузка и подготовка датасета

In [ ]:
%%time

print('Loading train features...')
df = pl.read_parquet(FEATURES_OUT / 'train_features.parquet')
print(f'Shape: {df.shape}')

# Labeled строки (target=0 или target=1)
df_labeled = df.filter(pl.col('target').is_not_null())
print(f'Labeled: {len(df_labeled):,}')
print('Target dist:', df_labeled['target'].value_counts().sort('target'))

# Null строки для сэмплирования негативов
df_null = df.filter(pl.col('target').is_null())
print(f'Null (green): {len(df_null):,}')

In [ ]:
def prepare_dataset(df_labeled: pl.DataFrame, df_null: pl.DataFrame,
                    null_ratio: float = 5.0, seed: int = 42) -> pl.DataFrame:
    """
    Формируем тренировочный датасет:
    - target=1: все fraud строки
    - target=0: все подтверждённые + сэмпл из null (null_ratio × n_fraud)
    """
    n_fraud = df_labeled.filter(pl.col('target') == 1).height
    n_null_sample = int(n_fraud * null_ratio)
    
    df_null_sample = df_null.sample(n=min(n_null_sample, len(df_null)), seed=seed)
    df_null_sample = df_null_sample.with_columns(pl.lit(0).cast(pl.Int32).alias('target'))
    
    result = pl.concat([df_labeled, df_null_sample])
    print(f'Dataset: {len(result):,} rows')
    print('  target=1:', result.filter(pl.col('target')==1).height)
    print('  target=0:', result.filter(pl.col('target')==0).height)
    return result

df_dataset = prepare_dataset(df_labeled, df_null, null_ratio=5.0)

## 2. TimeSeriesSplit — разбивка по времени

In [ ]:
def time_split(df: pl.DataFrame, val_from: str):
    """Разбивка: всё до val_from = train, от val_from = val."""
    val_dt = pl.lit(val_from).str.to_datetime('%Y-%m-%d')
    train = df.filter(pl.col('event_dttm') < val_dt)
    val = df.filter(pl.col('event_dttm') >= val_dt)
    print(f'Train: {len(train):,} | Val: {len(val):,}')
    print(f'  Train fraud: {train.filter(pl.col("target")==1).height} | Val fraud: {val.filter(pl.col("target")==1).height}')
    return train, val

def to_xy(df: pl.DataFrame, feature_cols: list):
    X = df.select(feature_cols).to_pandas()
    y = df['target'].to_numpy().astype(int)
    return X, y

# Используем последние 2 месяца как валидацию
df_tr, df_val = time_split(df_dataset, '2025-04-01')
X_train, y_train = to_xy(df_tr, FEATURE_COLS)
X_val, y_val = to_xy(df_val, FEATURE_COLS)

## 3. LightGBM GPU

In [ ]:
%%time

n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
scale_pos = n_neg / n_pos
print(f'scale_pos_weight: {scale_pos:.2f}')

LGBM_PARAMS = {
    'objective': 'binary',
    'metric': 'average_precision',
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    
    'scale_pos_weight': scale_pos,
    
    'n_estimators': 3000,
    'learning_rate': 0.05,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

lgbm_model = lgb.LGBMClassifier(**LGBM_PARAMS)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(150, verbose=True),
        lgb.log_evaluation(100),
    ]
)

lgbm_preds = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_score = average_precision_score(y_val, lgbm_preds)
print(f'\n=== LightGBM Val PR-AUC: {lgbm_score:.4f} ===')

lgbm_model.booster_.save_model(str(MODELS_OUT / 'lgbm.txt'))
print('Saved lgbm.txt')

## 4. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fi = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': lgbm_model.feature_importances_
}).sort_values('importance', descending=False).tail(30)

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(fi['feature'], fi['importance'])
ax.set_title('LightGBM Feature Importance (top 30)')
plt.tight_layout()
plt.savefig(ROOT / 'feature_importance.png', dpi=100)
plt.show()

## 5. CatBoost GPU

In [ ]:
%%time

cat_features_idx = []  # все фичи числовые после нашей обработки

cat_model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    task_type='GPU',
    devices='0',
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=200,
    early_stopping_rounds=150,
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
)

cat_preds = cat_model.predict_proba(X_val)[:, 1]
cat_score = average_precision_score(y_val, cat_preds)
print(f'\n=== CatBoost Val PR-AUC: {cat_score:.4f} ===')

cat_model.save_model(str(MODELS_OUT / 'catboost.cbm'))
print('Saved catboost.cbm')

## 6. Ансамбль (взвешенное среднее по PR-AUC)

In [ ]:
total = lgbm_score + cat_score
w_lgbm = lgbm_score / total
w_cat = cat_score / total

ensemble_preds = lgbm_preds * w_lgbm + cat_preds * w_cat
ensemble_score = average_precision_score(y_val, ensemble_preds)

print(f'LightGBM:  {lgbm_score:.4f} (w={w_lgbm:.2f})')
print(f'CatBoost:  {cat_score:.4f} (w={w_cat:.2f})')
print(f'Ensemble:  {ensemble_score:.4f}')

# Сохраняем веса
import json
with open(MODELS_OUT / 'weights.json', 'w') as f:
    json.dump({'lgbm': w_lgbm, 'catboost': w_cat}, f)
print('Saved weights.json')

## 7. Обучение финальной модели на ВСЕХ данных

In [ ]:
%%time
# После того как выбрали лучшие гиперпараметры — переобучаем на всём train

df_full_dataset = prepare_dataset(df_labeled, df_null, null_ratio=5.0)
X_full, y_full = to_xy(df_full_dataset, FEATURE_COLS)

# LightGBM — устанавливаем n_estimators = best_iteration * 1.1
best_iter = lgbm_model.best_iteration_
print(f'Best LGBM iteration: {best_iter}, training final with {int(best_iter * 1.1)}')

final_lgbm_params = {**LGBM_PARAMS, 'n_estimators': int(best_iter * 1.1)}
# Убираем early stopping для финального обучения
final_lgbm = lgb.LGBMClassifier(**final_lgbm_params)
final_lgbm.fit(X_full, y_full)
final_lgbm.booster_.save_model(str(MODELS_OUT / 'lgbm_final.txt'))
print('Saved lgbm_final.txt')